# 🔍 Hướng dẫn sử dụng `jdeprscan`

**jdeprscan** là công cụ đi kèm JDK (từ JDK 9+) dùng để quét các API **deprecated** (đã lỗi thời) trong file `.class` hoặc JAR. Rất hữu ích khi nâng cấp JDK để biết code nào đang dùng API sắp bị gỡ bỏ.

## Các cách gọi jdeprscan
| Lệnh | Mô tả |
|---|---|
| `jdeprscan --release 17 myapp.jar` | Quét JAR, kiểm tra deprecated API theo JDK 17 |
| `jdeprscan --release 17 --verbose myapp.jar` | Chi tiết hơn, liệt kê cả deprecated API không được dùng |
| `jdeprscan --list --release 17` | Liệt kê tất cả deprecated API trong JDK 17 |
| `jdeprscan --for-removal --release 17 myapp.jar` | Chỉ hiện API **sẽ bị gỡ hoàn toàn** (forRemoval=true) |

In [1]:
import subprocess
import shutil
import os
import json
from pathlib import Path

# ── 1. Tìm jdeprscan trên hệ thống ──
def find_jdeprscan() -> str | None:
    """Tìm executable jdeprscan trong PATH, JAVA_HOME, hoặc thư mục JDK phổ biến."""
    # Cách 1: tìm trong PATH
    direct = shutil.which("jdeprscan")
    if direct:
        return direct

    # Cách 2: tìm cạnh java executable
    java_exec = shutil.which("java")
    if java_exec:
        bin_dir = Path(java_exec).resolve().parent
        for name in ("jdeprscan.exe", "jdeprscan"):
            candidate = bin_dir / name
            if candidate.exists():
                return str(candidate)

    # Cách 3: dùng JAVA_HOME
    java_home = os.getenv("JAVA_HOME")
    if java_home:
        for name in ("jdeprscan.exe", "jdeprscan"):
            candidate = Path(java_home) / "bin" / name
            if candidate.exists():
                return str(candidate)

    # Cách 4: tìm trong thư mục JDK phổ biến trên Windows
    if os.name == "nt":
        common_roots = [
            os.getenv("ProgramFiles"),
            os.getenv("ProgramFiles(x86)"),
        ]
        for root in common_roots:
            if not root:
                continue
            for jdk_dir in Path(root).glob("Java/jdk*"):
                for name in ("jdeprscan.exe", "jdeprscan"):
                    candidate = jdk_dir / "bin" / name
                    if candidate.exists():
                        return str(candidate)

    return None

jdeprscan_path = find_jdeprscan()
print(f"jdeprscan found: {jdeprscan_path or '❌ NOT FOUND'}")

jdeprscan found: C:\Program Files\Java\jdk-17\bin\jdeprscan.exe


In [2]:
# ── 2. Liệt kê tất cả deprecated API trong JDK 17 ──
if jdeprscan_path:
    result = subprocess.run(
        [jdeprscan_path, "--list", "--release", "17"],
        capture_output=True, text=True, timeout=30
    )
    lines = result.stdout.strip().splitlines()
    print(f"Tổng số deprecated API trong JDK 17: {len(lines)}")
    print("\n📌 10 deprecated API đầu tiên:")
    for line in lines[:10]:
        print(f"  {line}")
else:
    print("SKIP: jdeprscan không khả dụng")

Tổng số deprecated API trong JDK 17: 658

📌 10 deprecated API đầu tiên:
  @Deprecated(since="1.7") javax.xml.stream.XMLEventFactory javax.xml.stream.XMLEventFactory.newInstance(java.lang.String,java.lang.ClassLoader)
  @Deprecated(since="1.7") javax.xml.stream.XMLInputFactory javax.xml.stream.XMLInputFactory.newInstance(java.lang.String,java.lang.ClassLoader)
  @Deprecated(since="1.7") javax.xml.stream.XMLInputFactory javax.xml.stream.XMLOutputFactory.newInstance(java.lang.String,java.lang.ClassLoader)
  @Deprecated javax.crypto.interfaces.DHPrivateKey.serialVersionUID
  @Deprecated javax.crypto.interfaces.DHPublicKey.serialVersionUID
  @Deprecated javax.crypto.interfaces.PBEKey.serialVersionUID
  @Deprecated(since="9") java.awt.Rectangle javax.swing.plaf.TextUI.modelToView(javax.swing.text.JTextComponent,int)
  @Deprecated(since="9") java.awt.Rectangle javax.swing.plaf.TextUI.modelToView(javax.swing.text.JTextComponent,int,javax.swing.text.Position.Bias)
  @Deprecated(since="9") int j

In [3]:
# ── 3. Chuẩn bị file JAR mẫu để quét ──
# Biên dịch và đóng gói lớp DeprecatedSample thành JAR
if jdeprscan_path:
    bin_dir = Path(jdeprscan_path).parent
    javac = bin_dir / ("javac.exe" if os.name == "nt" else "javac")
    jar_tool = bin_dir / ("jar.exe" if os.name == "nt" else "jar")

    smoke_dir = Path("artifacts/jdeprscan_smoke")
    java_file = smoke_dir / "DeprecatedSample.java"
    class_dir = smoke_dir / "classes"
    class_dir.mkdir(exist_ok=True)
    jar_file = smoke_dir / "deprecated-sample.jar"

    # Biên dịch
    compile_result = subprocess.run(
        [str(javac), "-d", str(class_dir), str(java_file)],
        capture_output=True, text=True
    )
    if compile_result.returncode == 0:
        print("✅ Biên dịch thành công")
        # Đóng gói JAR
        jar_result = subprocess.run(
            [str(jar_tool), "cf", str(jar_file), "-C", str(class_dir), "."],
            capture_output=True, text=True
        )
        print(f"✅ Tạo JAR: {jar_file} (exists={jar_file.exists()})")
    else:
        print(f"❌ Lỗi biên dịch: {compile_result.stderr}")
else:
    print("SKIP: jdeprscan không khả dụng")

✅ Biên dịch thành công
✅ Tạo JAR: artifacts\jdeprscan_smoke\deprecated-sample.jar (exists=True)


In [4]:
# ── 4. Chạy jdeprscan trên JAR mẫu ──
if jdeprscan_path and jar_file.exists():
    # Quét cơ bản
    result = subprocess.run(
        [jdeprscan_path, "--release", "17", str(jar_file)],
        capture_output=True, text=True, timeout=30
    )
    print("=== jdeprscan --release 17 deprecated-sample.jar ===")
    print(result.stdout or "(no stdout)")
    if result.stderr:
        print("STDERR:", result.stderr)

    print("\n--- Phân tích kết quả ---")
    for line in result.stdout.strip().splitlines():
        if "deprecated" in line.lower():
            print(f"⚠️  {line.strip()}")

=== jdeprscan --release 17 deprecated-sample.jar ===
Jar file artifacts\jdeprscan_smoke\deprecated-sample.jar:
class DeprecatedSample uses deprecated method java/lang/Thread::stop()V 


--- Phân tích kết quả ---
⚠️  Jar file artifacts\jdeprscan_smoke\deprecated-sample.jar:
⚠️  class DeprecatedSample uses deprecated method java/lang/Thread::stop()V


In [5]:
# ── 5. Dùng wrapper run_jdeprscan_check từ project ──
# Project MYGRATE đã có sẵn wrapper Python trong src/tools/maven_upgrade_tools.py
import sys
sys.path.insert(0, str(Path(".").resolve()))

from src.tools.maven_upgrade_tools import run_jdeprscan_check

jar_path = "artifacts/jdeprscan_smoke/deprecated-sample.jar"
if Path(jar_path).exists():
    report = run_jdeprscan_check(jar_path, target_release="17")
    print(json.dumps(report, indent=2, ensure_ascii=False))
else:
    print("SKIP: JAR chưa được tạo")

{
  "status": "WARN",
  "executable": "C:\\Program Files\\Java\\jdk-17\\bin\\jdeprscan.exe",
  "exit_code": 0,
  "finding_count": 1,
  "findings": [
    {
      "line": "class DeprecatedSample uses deprecated method java/lang/Thread::stop()V"
    }
  ],
  "output": "Jar file artifacts/jdeprscan_smoke/deprecated-sample.jar:\nclass DeprecatedSample uses deprecated method java/lang/Thread::stop()V"
}


## 📝 Tóm tắt

| Bước | Lệnh / Hàm | Mục đích |
|------|-----------|----------|
| Tìm tool | `find_jdeprscan()` | Tìm `jdeprscan` trên hệ thống |
| Liệt kê API | `jdeprscan --list --release 17` | Xem tất cả deprecated API trong JDK 17 |
| Quét JAR | `jdeprscan --release 17 app.jar` | Phát hiện code dùng deprecated API |
| Quét chi tiết | `jdeprscan --release 17 --verbose app.jar` | Bao gồm cả deprecated API không được dùng |
| Chỉ forRemoval | `jdeprscan --for-removal --release 17 app.jar` | Chỉ hiện API **sẽ bị gỡ hoàn toàn** |
| Wrapper Python | `run_jdeprscan_check(jar, release)` | Dùng từ code Python, trả dict có status/findings |

> ⚠️ `jdeprscan` chỉ phân tích **bytecode** (không cần source code). Nó phát hiện API deprecated ở **thời điểm biên dịch**, không phải runtime.

# 🧪 Dùng jdeprscan khoanh vùng code cần upgrade — Case study: Cantor
## Workflow: compile JDK 8 -> scan jdeprscan JDK 17 -> khoanh vùng -> quyết định upgrade
## Tại sao 2 JDK?
- Cantor pom.xml phụ thuộc jdk.tools (tools.jar) -> chỉ JDK 8 mới có
- jdeprscan cần bytecode đã compile -> phải compile trước
- jdeprscan --release 17 check bytecode theo deprecated list của JDK 17

In [6]:
CANTOR_PATH = Path(r"D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor")

# ── Dual-JDK: JDK 8 compile, JDK 17 scan ──
# Cantor phụ thuộc jdk.tools (tools.jar) nên bắt buộc compile bằng JDK 8
# jdeprscan thì phải dùng JDK 17 vì mình đang check upgrade lên 17
JDK8_HOME = Path(os.getenv("LOCALAPPDATA", "")) / "Java" / "jdk-8"
JDK17_HOME = Path(jdeprscan_path).parent.parent

jdk8_javac = JDK8_HOME / "bin" / ("javac.exe" if os.name == "nt" else "javac")
jdk8_jar   = JDK8_HOME / "bin" / ("jar.exe" if os.name == "nt" else "jar")
jdk8_tools = JDK8_HOME / "lib" / "tools.jar"

print(f"JDK 8  (compile):   {JDK8_HOME}  javac={jdk8_javac.exists()}")
print(f"JDK 17 (jdeprscan): {JDK17_HOME}  jdeprscan={Path(jdeprscan_path).exists()}")
print(f"tools.jar: {jdk8_tools.exists()}")

if not jdk8_javac.exists():
    print("!! JDK 8 javac không tìm thấy -> không compile được Cantor")
    print("   Cài: https://adoptium.net/temurin/releases/?version=8")

# ── helper: nối classpath từ thư mục JAR ──
def build_classpath(jar_dir: Path) -> tuple[list[str], str]:
    jars = sorted(str(p) for p in jar_dir.glob("*.jar"))
    sep = ";" if os.name == "nt" else ":"
    return jars, sep.join(jars)

# ── helper: chạy jdeprscan trên 1 JAR, trả dict ──
def scan_jar(jdeprscan: str, jar_path: str, release: str = "17") -> dict:
    r = subprocess.run(
        [jdeprscan, "--release", release, jar_path],
        capture_output=True, text=True, timeout=60
    )
    raw = (r.stdout + r.stderr).strip()
    lines = raw.splitlines() if raw else []
    total = sum(1 for l in lines if "deprecated" in l.lower() or "forRemoval" in l.lower())
    for_removal = sum(1 for l in lines if "forRemoval=true" in l)
    return {"jar": Path(jar_path).name, "total": total, "for_removal": for_removal, "lines": lines}

# ── helper: progress bar đơn giản ──
def progress(current, total, width=30):
    pct = current / total if total else 1
    bar = "█" * int(width * pct) + "░" * (width - int(width * pct))
    return f"[{bar}] {current}/{total} ({pct:.0%})"

# ══════════════════════════════════════════════════════════════
# B0: Maven resolve dependencies
# Chạy Maven với JAVA_HOME=JDK8 vì pom.xml cần jdk.tools
# ══════════════════════════════════════════════════════════════
print("\n── B0: Maven resolve dependencies ──")

env = os.environ.copy()
env["JAVA_HOME"] = str(JDK8_HOME)

mvn = shutil.which("mvn") or shutil.which("mvn.cmd")
if not mvn:
    local_maven = Path(os.getenv("LOCALAPPDATA", "")) / "Apache"
    if local_maven.exists():
        maven_dirs = sorted(local_maven.glob("apache-maven-*"))
        if maven_dirs:
            mvn_cmd = maven_dirs[-1] / "bin" / "mvn.cmd"
            if mvn_cmd.exists():
                mvn = str(mvn_cmd)

dep_dir = CANTOR_PATH / "target" / "dependency"

if mvn:
    print(f"  mvn: {mvn}")
    print(f"  JAVA_HOME: {env['JAVA_HOME']}")
    print(f"  đang resolve... ", end="", flush=True)

    dep_result = subprocess.run(
        [mvn, "dependency:copy-dependencies",
         "-DincludeScope=runtime",
         "-DoutputDirectory=" + str(dep_dir),
         "-f", str(CANTOR_PATH / "pom.xml")],
        capture_output=True, text=True, timeout=180, env=env
    )
    if dep_result.returncode == 0:
        n = len(list(dep_dir.glob("*.jar")))
        print(f"OK ({n} JARs)")
    else:
        print("FAIL")
        print(dep_result.stderr[-300:] if dep_result.stderr else dep_result.stdout[-300:])
else:
    print("  !! Maven không tìm thấy")

# build classpath
if dep_dir.exists() and any(dep_dir.glob("*.jar")):
    classpath_jars, cp = build_classpath(dep_dir)
    dep_source = str(dep_dir)
else:
    fallback_dir = Path("runtime_test_jars")
    classpath_jars, cp = build_classpath(fallback_dir)
    dep_source = str(fallback_dir) + " (fallback, không phải dependency thật)"

print(f"  classpath: {len(classpath_jars)} JARs từ {dep_source}")

# ══════════════════════════════════════════════════════════════
# B1: Compile Cantor bằng JDK 8
# Chỉ cần bytecode cho jdeprscan, không phải build production
# ══════════════════════════════════════════════════════════════
print("\n── B1: Compile Cantor (JDK 8) ──")

src_dir = CANTOR_PATH / "src" / "main" / "java"
out_dir = CANTOR_PATH / "target" / "classes"
out_dir.mkdir(parents=True, exist_ok=True)
java_files = list(src_dir.rglob("*.java"))

# chèn tools.jar vào classpath (Cantor pom.xml khai báo jdk.tools)
compile_cp = cp
if jdk8_tools.exists() and str(jdk8_tools) not in compile_cp:
    sep = ";" if os.name == "nt" else ":"
    compile_cp = compile_cp + sep + str(jdk8_tools)

print(f"  {len(java_files)} source files, compiler: {jdk8_javac}")
print(f"  compiling... ", end="", flush=True)

comp = subprocess.run(
    [str(jdk8_javac), "-cp", compile_cp, "-d", str(out_dir)] + [str(f) for f in java_files],
    capture_output=True, text=True, timeout=60
)
if comp.returncode == 0:
    n_class = len(list(out_dir.rglob("*.class")))
    print(f"OK ({n_class} class files)")
else:
    print(f"FAIL\n{comp.stderr[:500]}")

# đóng gói JAR
cantor_jar = CANTOR_PATH / "target" / "cantor.jar"
if out_dir.exists() and any(out_dir.rglob("*.class")):
    subprocess.run(
        [str(jdk8_jar), "cf", str(cantor_jar), "-C", str(out_dir), "."],
        capture_output=True, text=True
    )
print(f"  cantor.jar: {cantor_jar.exists()}")

# ══════════════════════════════════════════════════════════════
# B2: jdeprscan Cantor JAR
# Bytecode JDK 8, scan theo JDK 17 deprecated list
# ══════════════════════════════════════════════════════════════
print("\n── B2: jdeprscan Cantor JAR (JDK 17) ──")

for_removal = []      # khởi tạo trước, khỏi sợ NameError
deprecated_only = []

if jdeprscan_path and cantor_jar.exists():
    print(f"  scanning... ", end="", flush=True)
    scan_result = subprocess.run(
        [jdeprscan_path, "--release", "17", "--class-path", compile_cp, str(cantor_jar)],
        capture_output=True, text=True, timeout=60
    )
    output = (scan_result.stdout + scan_result.stderr).strip()

    if output:
        for line in output.splitlines():
            if "forRemoval=true" in line:
                for_removal.append(line.strip())
            elif "deprecated" in line.lower() or "uses deprecated" in line.lower():
                deprecated_only.append(line.strip())
        print(f"found {len(for_removal)} forRemoval, {len(deprecated_only)} deprecated")
        if for_removal:
            print("  !! forRemoval (sẽ gỡ hẳn):")
            for l in for_removal:
                print(f"     {l}")
        if deprecated_only:
            print("  ! deprecated (chưa gỡ):")
            for l in deprecated_only:
                print(f"     {l}")
        if not for_removal and not deprecated_only:
            print("  -> Cantor code sạch, không dùng deprecated API nào")
    else:
        print("clean — Cantor không dùng deprecated API nào trong JDK 17")
else:
    print("  SKIP: jdeprscan hoặc cantor.jar không có")

# ══════════════════════════════════════════════════════════════
# B3: jdeprscan từng dependency JAR
# Đây là phần quan trọng nhất — khoanh vùng xem dependency nào đang kéo
# deprecated API vào project
# ══════════════════════════════════════════════════════════════
print("\n── B3: jdeprscan dependencies ──")

if jdeprscan_path:
    scan_dir = dep_dir if (dep_dir.exists() and any(dep_dir.glob("*.jar"))) else Path("runtime_test_jars")
    all_dep_jars = sorted(scan_dir.glob("*.jar"))
    total_jars = len(all_dep_jars)
    print(f"  {total_jars} JARs từ {scan_dir}")

    summary = []
    for i, hj in enumerate(all_dep_jars, 1):
        # progress — in trên 1 dòng, ghi đè liên tục
        print(f"\r  {progress(i, total_jars)} {hj.name[:40]:<40}", end="", flush=True)
        info = scan_jar(jdeprscan_path, str(hj))
        summary.append(info)

    print()  # xuong dong sau progress

    # in kết quả — chỉ hiện JARs có vấn đề
    problem_jars = [s for s in summary if s["total"] > 0]
    clean_jars = [s for s in summary if s["total"] == 0]
    print(f"\n  {len(problem_jars)} JARs có deprecated API, {len(clean_jars)} JARs sạch")

    if problem_jars:
        # xếp hạng: forRemoval nhiều nhất lên đầu
        ranked = sorted(problem_jars, key=lambda s: (s["for_removal"], s["total"]), reverse=True)

        print("\n  -- Top nghiêm trọng (forRemoval nhiều nhất) --")
        for s in ranked[:15]:
            tag = "!!" if s["for_removal"] >= 10 else "!" if s["for_removal"] > 0 else " "
            print(f"  {tag} {s['jar']:<40} forRemoval={s['for_removal']:>3}  total={s['total']:>3}")
        if len(ranked) > 15:
            print(f"  ... +{len(ranked)-15} JARs khác")

        # phân nhóm theo ecosystem — cho thấy vấn đề không chỉ Hadoop
        print("\n  -- Phân nhóm theo ecosystem --")
        eco_rules = {
            "JDK internals":  lambda n: "jdk.tools" in n,
            "Hadoop":         lambda n: "hadoop" in n,
            "Commons":        lambda n: n.startswith("commons-") or "beanutils" in n,
            "Java EE / JAXB": lambda n: any(k in n for k in ("jaxb", "activation", "servlet", "jsp", "jersey", "jetty", "jasper")),
            "Logging":        lambda n: any(k in n for k in ("log4j", "slf4j", "commons-logging")),
            "Serialization":  lambda n: any(k in n for k in ("jackson", "avro", "jettison", "protobuf", "snappy")),
        }
        ecosystems = {}
        assigned = set()
        for eco_name, rule in eco_rules.items():
            members = [s for s in problem_jars if rule(s["jar"])]
            if members:
                ecosystems[eco_name] = members
                assigned.update(s["jar"] for s in members)
        leftovers = [s for s in problem_jars if s["jar"] not in assigned]
        if leftovers:
            ecosystems["Khác"] = leftovers

        for eco_name, members in ecosystems.items():
            fr = sum(s["for_removal"] for s in members)
            tot = sum(s["total"] for s in members)
            tag = "!!" if fr >= 10 else "!" if fr > 0 else " "
            print(f"  {tag} {eco_name:<18} {len(members):>2} JARs, forRemoval={fr:>3}, total={tot:>3}")

    # ══════════════════════════════════════════════════════════
    # Tổng hợp: 3 lớp cần xử lý
    # ══════════════════════════════════════════════════════════
    print(f"\n{'─'*55}")
    print("TỔNG HỢP — 3 lớp cần xử lý khi upgrade JDK 17")
    print(f"{'─'*55}")

    # lớp 1: Cantor code
    print(f"\n  1. Cantor code:")
    if len(for_removal) > 0:
        print(f"     !! {len(for_removal)} API forRemoval -> phải sửa, không thì crash runtime")
    if len(deprecated_only) > 0:
        print(f"     !  {len(deprecated_only)} API deprecated -> nên sửa, vẫn chạy nhưng cảnh báo")
    if not for_removal and not deprecated_only:
        print(f"     -> Sạch, không cần sửa gì")

    # lớp 2: dependencies
    print(f"\n  2. Dependencies:")
    if problem_jars:
        top3 = ranked[:3]
        for s in top3:
            print(f"     !! {s['jar']}: {s['for_removal']} forRemoval / {s['total']} total")
        print(f"     -> Nâng version hoặc thay lib, không chỉ Hadoop đâu")
        print(f"     -> jdk.tools, commons-beanutils, log4j, jasper... đều phải xử lý")
    else:
        print(f"     -> Không có dependency nào dùng deprecated API")

    # lớp 3: pom.xml
    print(f"\n  3. pom.xml:")
    print(f"     !  hadoop 2.2.0 + jdk.tools (tools.jar)")
    print(f"     -> jdk.tools bị gỡ từ JDK 9, phải bỏ hoặc thay thế")
    print(f"     -> Hadoop 2.2.0 quá cũ, nên nâng lên 3.x")

    print(f"\n  Workflow: compile JDK 8 -> scan jdeprscan JDK 17")
    print(f"  JDK 8:  {JDK8_HOME}")
    print(f"  JDK 17: {JDK17_HOME}")

JDK 8  (compile):   C:\Users\tngtr\AppData\Local\Java\jdk-8  javac=True
JDK 17 (jdeprscan): C:\Program Files\Java\jdk-17  jdeprscan=True
tools.jar: True

── B0: Maven resolve dependencies ──
  mvn: C:\Users\tngtr\AppData\Local\Apache\apache-maven-3.9.16\bin\mvn.cmd
  JAVA_HOME: C:\Users\tngtr\AppData\Local\Java\jdk-8
  đang resolve... OK (54 JARs)
  classpath: 54 JARs từ D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor\target\dependency

── B1: Compile Cantor (JDK 8) ──
  3 source files, compiler: C:\Users\tngtr\AppData\Local\Java\jdk-8\bin\javac.exe
  compiling... OK (2 class files)
  cantor.jar: True

── B2: jdeprscan Cantor JAR (JDK 17) ──
  scanning... found 0 forRemoval, 0 deprecated
  -> Cantor code sạch, không dùng deprecated API nào

── B3: jdeprscan dependencies ──
  54 JARs từ D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor\target\dependency
  [██████████████████████████████] 54/54 (100%) zookeeper-3.4.5.jar                     

  35 